<a href="https://colab.research.google.com/github/SamanTarique/flyrank-01-ml-2026/blob/main/work/notebooks/w04_1baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:


%pip -q install duckdb Huggingface_hub


In [2]:
import os ,getpass

HF_token=os.environ.get('saman_tech')
if not HF_token:
    try:
        from google.colab import userdata
        HF_token = userdata.get('saman_tech')
    except Exception:
        pass
HF_token = HF_token or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

files_list_df = con.sql(f"""
    SELECT file
    FROM glob('{REL}/*.parquet')
""").df()

print("--- Data Warehouse Files ---")
for file_name in files_list_df['file']:
    print(file_name)


--- Data Warehouse Files ---
hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet


In [3]:
selected_files = files_list_df[
    files_list_df['file'].isin([
        f"{REL}/dim_content.parquet",
        f"{REL}/fact_content_query_90d.parquet"
    ])
]['file'].tolist()

print(selected_files)

['hf://datasets/FlyRank/internship-warehouse/dim_content.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet']


In [4]:
dataframes = {}

for file in selected_files:
    df = con.sql(f"SELECT * FROM '{file}'").df()
    name = file.split("/")[-1].replace(".parquet", "")
    dataframes[name] = df

    print(f"{name}: {df.shape}")

dim_content: (519606, 26)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_query_90d: (2414248, 21)


In [5]:
for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


--- dim_content ---
Shape: (519606, 26)
Columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

--- fact_content_query_90d ---
Shape: (2414248, 21)
Columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'ano

### Missing Values Calculation (dim_content)
We already calculated the missing values for the `dim_content` file as part of the Week 3 tasks, so we are skipping the recalculation for Week 4. For reference, here is the summary of the missing values we found:

| Metric | Count |
| :--- | :--- |
| **Total Rows** | 519,606 |
| **Missing `char_count`** | 177,768 |
| **Missing `last_optimized_date`** | 474,210 |
| **Missing `content_type`** | 0 |

In [6]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(content_total_impressions_90d) AS missing_content_total_impressions_90d,
    COUNT(*) - COUNT(impressions_90d) AS missing_impressions_90d,
    COUNT(*) - COUNT(avg_position_90d) AS missing_avg_position_90d,
    COUNT(*) - COUNT(content_visible_query_count) AS missing_content_visible_query_count,
    COUNT(*) - COUNT(rare_query_count) AS missing_rare_query_count
FROM '{REL}/fact_content_query_90d.parquet'
""").df()

,total_rows,missing_content_total_impressions_90d,missing_impressions_90d,missing_avg_position_90d,missing_content_visible_query_count,missing_rare_query_count
0,2414248,0,0,0,0,0


In [7]:
con.sql(f"""
SELECT
    MIN(content_total_impressions_90d) AS min_volume,
    MAX(content_total_impressions_90d) AS max_volume,
    AVG(content_total_impressions_90d) AS avg_volume,
    MEDIAN(content_total_impressions_90d) AS median_volume
FROM '{REL}/fact_content_query_90d.parquet'
""").df()

,min_volume,max_volume,avg_volume,median_volume
0,10,1961517,44466.668938,12942.0


In [8]:
con.sql(f"""
SELECT
    QUANTILE_CONT(content_total_impressions_90d, 0.25) AS p25,
    QUANTILE_CONT(content_total_impressions_90d, 0.50) AS p50,
    QUANTILE_CONT(content_total_impressions_90d, 0.75) AS p75,
    QUANTILE_CONT(content_total_impressions_90d, 0.90) AS p90
FROM '{REL}/fact_content_query_90d.parquet'
""").df()

,p25,p50,p75,p90
0,3894.0,12942.0,38211.0,94638.0


In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(avg_position_last30) AS missing_last30,
    COUNT(*) - COUNT(avg_position_prev30) AS missing_prev30,
    COUNT(*) FILTER (
        WHERE avg_position_last30 IS NOT NULL
          AND avg_position_prev30 IS NOT NULL
    ) AS usable_for_declined
FROM '{REL}/fact_content_query_90d.parquet'
""").df()

,total_rows,missing_last30,missing_prev30,usable_for_declined
0,2414248,530758,329420,1704767


#### Volume bucket table

In [10]:
con.sql(f"""
SELECT
    CASE
        WHEN content_total_impressions_90d <= 3894 THEN 'Low'
        WHEN content_total_impressions_90d <= 12942 THEN 'Medium'
        WHEN content_total_impressions_90d <= 38211 THEN 'High'
        ELSE 'Very High'
    END AS volume_bucket,
    COUNT(*) AS n
FROM '{REL}/fact_content_query_90d.parquet'
GROUP BY volume_bucket
ORDER BY
    CASE volume_bucket
        WHEN 'Low' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'High' THEN 3
        WHEN 'Very High' THEN 4
    END
""").df()

,volume_bucket,n
0,Low,603673
1,Medium,603452
2,High,603613
3,Very High,603510


In [11]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE content_total_impressions_90d >= 500
    ) AS volume_500_plus,

    COUNT(*) FILTER (
        WHERE content_total_impressions_90d >= 1000
    ) AS volume_1000_plus,

    COUNT(*) FILTER (
        WHERE content_total_impressions_90d >= 5000
    ) AS volume_5000_plus,

    COUNT(*) FILTER (
        WHERE content_total_impressions_90d >= 10000
    ) AS volume_10000_plus,

    COUNT(*) FILTER (
        WHERE content_total_impressions_90d >= 50000
    ) AS volume_50000_plus

FROM '{REL}/fact_content_query_90d.parquet'
""").df()

,total_rows,volume_500_plus,volume_1000_plus,volume_5000_plus,volume_10000_plus,volume_50000_plus
0,2414248,2321566,2215563,1705945,1355368,476142


In [12]:
con.sql(f"""
SELECT
    CASE
        WHEN content_total_impressions_90d <= 3894 THEN 'Low'
        WHEN content_total_impressions_90d <= 12942 THEN 'Medium'
        WHEN content_total_impressions_90d <= 38211 THEN 'High'
        ELSE 'Very High'
    END AS volume_bucket,

    COUNT(*) AS n,

    ROUND(AVG(content_total_impressions_90d), 2) AS avg_volume,

    ROUND(AVG(avg_position_90d), 2) AS avg_position

FROM '{REL}/fact_content_query_90d.parquet'

GROUP BY volume_bucket

ORDER BY
    CASE volume_bucket
        WHEN 'Low' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'High' THEN 3
        WHEN 'Very High' THEN 4
    END
""").df()

,volume_bucket,n,avg_volume,avg_position
0,Low,603673,1711.43,32.28
1,Medium,603452,7819.07,18.14
2,High,603613,22900.33,15.83
3,Very High,603510,145447.55,12.81


#### Signal 2:Satlnesss

In [13]:
con.sql(f"""
SELECT
    MIN(last_optimized_date) AS earliest_optimized,
    MAX(last_optimized_date) AS latest_optimized,
    COUNT(*) AS total_rows,
    COUNT(last_optimized_date) AS rows_with_date
FROM '{REL}/dim_content.parquet'
WHERE last_optimized_date IS NOT NULL
""").df()

,earliest_optimized,latest_optimized,total_rows,rows_with_date
0,2026-04-24,2026-07-06,45396,45396


In [14]:
con.sql(f"""
SELECT
    MIN(last_optimized_date) AS earliest_optimized,
    MAX(last_optimized_date) AS latest_optimized,
    COUNT(*) FILTER (
        WHERE last_optimized_date > DATE '2026-06-30'
    ) AS dates_after_cutoff
FROM '{REL}/dim_content.parquet'
WHERE last_optimized_date IS NOT NULL
""").df()

,earliest_optimized,latest_optimized,dates_after_cutoff
0,2026-04-24,2026-07-06,4964


We discovered a temporal data issue: 4,964 non-missing last_optimized_date values occur after the 2026-06-30 cutoff. Using the cutoff to calculate staleness for these rows would produce negative values, so we should not silently clip or fill them; they need to be excluded from the pre-cutoff staleness audit.

In [15]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_with_date_to_cutoff,
    MIN(last_optimized_date) AS earliest_date_to_cutoff,
    MAX(last_optimized_date) AS latest_date_to_cutoff
FROM '{REL}/dim_content.parquet'
WHERE last_optimized_date IS NOT NULL
  AND last_optimized_date <= DATE '2026-06-30'
""").df()

,rows_with_date_to_cutoff,earliest_date_to_cutoff,latest_date_to_cutoff
0,40432,2026-04-24,2026-06-30


In [16]:
con.sql(f"""
SELECT
    CASE
        WHEN date_diff('day', last_optimized_date, DATE '2026-06-30') <= 30
            THEN '0-30 days'
        WHEN date_diff('day', last_optimized_date, DATE '2026-06-30') <= 60
            THEN '31-60 days'
        WHEN date_diff('day', last_optimized_date, DATE '2026-06-30') <= 90
            THEN '61-90 days'
        ELSE '90+ days'
    END AS staleness_bucket,
    COUNT(*) AS n
FROM '{REL}/dim_content.parquet'
WHERE last_optimized_date IS NOT NULL
  AND last_optimized_date <= DATE '2026-06-30'
GROUP BY staleness_bucket
ORDER BY
    CASE staleness_bucket
        WHEN '0-30 days' THEN 1
        WHEN '31-60 days' THEN 2
        WHEN '61-90 days' THEN 3
        WHEN '90+ days' THEN 4
    END
""").df()

,staleness_bucket,n
0,0-30 days,22721
1,31-60 days,17099
2,61-90 days,612


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
print("""Rule: Combine search volume and content staleness into one score to rank refresh opportunities.
Reason code: REFRESH_OPPORTUNITY
Action: REFRESH""")

Rule: Combine search volume and content staleness into one score to rank refresh opportunities.
Reason code: REFRESH_OPPORTUNITY
Action: REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
queue_df = con.sql(f"""
SELECT
    dc.content_hash_id,
    MAX(fcq.content_total_impressions_90d) AS content_total_impressions_90d,
    dc.last_optimized_date,

    date_diff(
        'day',
        dc.last_optimized_date,
        DATE '2026-06-30'
    ) AS staleness_days

FROM '{REL}/dim_content.parquet' AS dc

JOIN '{REL}/fact_content_query_90d.parquet' AS fcq
    ON dc.content_hash_id = fcq.content_hash_id

WHERE dc.last_optimized_date IS NOT NULL
  AND dc.last_optimized_date <= DATE '2026-06-30'

GROUP BY
    dc.content_hash_id,
    dc.last_optimized_date
""").df()

print("Rows available for baseline:", len(queue_df))
print("Unique content:", queue_df["content_hash_id"].nunique())
display(queue_df.head())

Rows available for baseline: 36293
Unique content: 36293


,content_hash_id,content_total_impressions_90d,last_optimized_date,staleness_days
0,content_4483e354a2992497,2169,2026-06-15,15
1,content_4486e5efcc7b773f,2311,2026-05-20,41
2,content_44c082bdb9a864ea,17583,2026-05-20,41
3,content_44e29d7baeb1cab0,4699,2026-06-11,19
4,content_451aa32a0eda305b,1776,2026-06-15,15


In [19]:
queue_df["volume_score"] = queue_df["content_total_impressions_90d"] / queue_df["content_total_impressions_90d"].max()
queue_df["staleness_score"] = queue_df["staleness_days"] / queue_df["staleness_days"].max()

display(queue_df.head())

,content_hash_id,content_total_impressions_90d,last_optimized_date,staleness_days,volume_score,staleness_score
0,content_4483e354a2992497,2169,2026-06-15,15,0.001106,0.223881
1,content_4486e5efcc7b773f,2311,2026-05-20,41,0.001178,0.611940
2,content_44c082bdb9a864ea,17583,2026-05-20,41,0.008964,0.611940
3,content_44e29d7baeb1cab0,4699,2026-06-11,19,0.002396,0.283582
4,content_451aa32a0eda305b,1776,2026-06-15,15,0.000905,0.223881


In [20]:
queue_df["score"] = 0.5 * queue_df["volume_score"] + 0.5 * queue_df["staleness_score"]
queue_df["reason_code"] = "REFRESH_OPPORTUNITY"
queue_df["action"] = "REFRESH"

queue_df = queue_df.sort_values("score", ascending=False).reset_index(drop=True)
queue_df["rank"] = queue_df.index + 1

display(queue_df.head(10))

,content_hash_id,content_total_impressions_90d,last_optimized_date,staleness_days,volume_score,staleness_score,score,reason_code,action,rank
0,content_eadb33b5df496f4a,1961517,2026-05-20,41,1.000000,0.611940,0.805970,REFRESH_OPPORTUNITY,REFRESH,1
1,content_545bb6cc7081ded3,984424,2026-05-20,41,0.501869,0.611940,0.556905,REFRESH_OPPORTUNITY,REFRESH,2
2,content_f7cab77acfb810fe,5308,2026-04-24,67,0.002706,1.000000,0.501353,REFRESH_OPPORTUNITY,REFRESH,3
3,content_8609e82e94db55e7,477,2026-04-24,67,0.000243,1.000000,0.500122,REFRESH_OPPORTUNITY,REFRESH,4
4,content_7d206261a67b9c81,973,2026-04-28,63,0.000496,0.940299,0.470397,REFRESH_OPPORTUNITY,REFRESH,5
5,content_c0a27325da0a4430,3775,2026-04-29,62,0.001925,0.925373,0.463649,REFRESH_OPPORTUNITY,REFRESH,6
6,content_11004a8bd822859c,3325,2026-04-29,62,0.001695,0.925373,0.463534,REFRESH_OPPORTUNITY,REFRESH,7
7,content_b9209d1acd0032c2,2920,2026-04-29,62,0.001489,0.925373,0.463431,REFRESH_OPPORTUNITY,REFRESH,8
8,content_00e646b19859dd43,2738,2026-04-29,62,0.001396,0.925373,0.463384,REFRESH_OPPORTUNITY,REFRESH,9
9,content_6572843a3bc99ac8,2139,2026-04-29,62,0.001090,0.925373,0.463232,REFRESH_OPPORTUNITY,REFRESH,10


In [21]:
import os

output = "work/outputs/baseline_action_score.csv"

os.makedirs(os.path.dirname(output), exist_ok=True)
queue_df.to_csv(output, index=False)

print(f"Saved: {output}")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
top20 = queue_df.head(20)[
    ["rank", "content_hash_id", "score", "reason_code", "action",
     "content_total_impressions_90d", "staleness_days"]
].copy()

top20["confidence_note"] = "Score is based on observed volume and staleness."
top20["what_would_make_it_wrong"] = "The content may not actually need a refresh."
display(top20.style.hide(axis="index"))


rank,content_hash_id,score,reason_code,action,content_total_impressions_90d,staleness_days,confidence_note,what_would_make_it_wrong
1,content_eadb33b5df496f4a,0.805970,REFRESH_OPPORTUNITY,REFRESH,1961517,41,Score is based on observed volume and staleness.,The content may not actually need a refresh.
2,content_545bb6cc7081ded3,0.556905,REFRESH_OPPORTUNITY,REFRESH,984424,41,Score is based on observed volume and staleness.,The content may not actually need a refresh.
3,content_f7cab77acfb810fe,0.501353,REFRESH_OPPORTUNITY,REFRESH,5308,67,Score is based on observed volume and staleness.,The content may not actually need a refresh.
4,content_8609e82e94db55e7,0.500122,REFRESH_OPPORTUNITY,REFRESH,477,67,Score is based on observed volume and staleness.,The content may not actually need a refresh.
5,content_7d206261a67b9c81,0.470397,REFRESH_OPPORTUNITY,REFRESH,973,63,Score is based on observed volume and staleness.,The content may not actually need a refresh.
6,content_c0a27325da0a4430,0.463649,REFRESH_OPPORTUNITY,REFRESH,3775,62,Score is based on observed volume and staleness.,The content may not actually need a refresh.
7,content_11004a8bd822859c,0.463534,REFRESH_OPPORTUNITY,REFRESH,3325,62,Score is based on observed volume and staleness.,The content may not actually need a refresh.
8,content_b9209d1acd0032c2,0.463431,REFRESH_OPPORTUNITY,REFRESH,2920,62,Score is based on observed volume and staleness.,The content may not actually need a refresh.
9,content_00e646b19859dd43,0.463384,REFRESH_OPPORTUNITY,REFRESH,2738,62,Score is based on observed volume and staleness.,The content may not actually need a refresh.
10,content_6572843a3bc99ac8,0.463232,REFRESH_OPPORTUNITY,REFRESH,2139,62,Score is based on observed volume and staleness.,The content may not actually need a refresh.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
import pandas as pd



print("WEAK PICKS")
display(
    top20[top20["content_total_impressions_90d"] < 1000][
        ["rank", "content_hash_id", "score",
         "content_total_impressions_90d", "staleness_days"]
    ].style.hide(axis="index")
)

print("\nLEAKAGE CHECK")
print("Future optimization dates:",
      (queue_df["last_optimized_date"] > pd.Timestamp("2026-06-30")).sum())

print("Negative staleness:",
      (queue_df["staleness_days"] < 0).sum())

print("\nPRODUCT-RELATED COLUMNS")
product_cols = [c for c in queue_df.columns if "product" in c.lower()]
print(product_cols if product_cols else "No product flag included in baseline data.")

WEAK PICKS


rank,content_hash_id,score,content_total_impressions_90d,staleness_days
4,content_8609e82e94db55e7,0.500122,477,67
5,content_7d206261a67b9c81,0.470397,973,63



LEAKAGE CHECK
Future optimization dates: 0
Negative staleness: 0

PRODUCT-RELATED COLUMNS
No product flag included in baseline data.


In [24]:
print("""Weak picks: Ranks 4 and 5 look weaker because their search volume is low, so their high scores are mainly driven by staleness.
Leakage check: No future optimization dates or negative staleness values were found, and no product flags were included in the baseline data.""")

Weak picks: Ranks 4 and 5 look weaker because their search volume is low, so their high scores are mainly driven by staleness.
Leakage check: No future optimization dates or negative staleness values were found, and no product flags were included in the baseline data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.